In [6]:
import time
import pandas as pd
import winsound # Thư viện tạo tiếng Bíp báo hiệu trên Windows
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.options import Options

In [7]:
def crawl_1000_reviews(driver, url, address_name):
    print(f"\n[{address_name}] >>> Đang mở link...")
    try:
        driver.get(url)
    except Exception as e:
        print(f"Lỗi truy cập: {e}")
        return []

    time.sleep(6) 
    
    try:
        tab = driver.find_element(By.XPATH, '//button[.//div[contains(text(), "Bài đánh giá") or contains(text(), "Reviews")]]')
        tab.click()
        time.sleep(3)
    except:
        pass

    try: winsound.Beep(2000, 1000)
    except: pass
        
    print("\n" + "="*60)
    print(f"🚨 ĐÃ MỞ XONG LINK: {address_name}")
    print("👉 BƯỚC 1: Hãy qua cửa sổ Chrome, tự chọn bộ lọc (Mới nhất/Cao nhất/Thấp nhất).")
    print("👉 BƯỚC 2: Quay lại đây BẤM PHÍM ENTER để code quét.")
    print("="*60)
    
    input("Bấm ENTER tại đây khi bạn đã sẵn sàng...") 

    collected_data = []
    seen_texts = set()
    stuck_counter = 0
    last_total = 0

    print(f"\n🚀 Bắt đầu càn quét mục tiêu 1000 đánh giá (Chống lỗi '...', CẤM DỊCH)...")

    while len(collected_data) < 1000: 
        items = driver.find_elements(By.CSS_SELECTOR, 'div[data-review-id]')
        if items:
            try:
                last_item = items[-1]
                driver.execute_script("arguments[0].scrollIntoView(true);", last_item)
                time.sleep(1)
                action = webdriver.ActionChains(driver)
                action.move_to_element(last_item).click().perform() 
                for _ in range(3):
                    action.send_keys(Keys.PAGE_DOWN).perform()
                    time.sleep(0.5)
            except:
                driver.execute_script("""
                    let mainDivs = document.querySelectorAll('div[role="main"]');
                    if(mainDivs.length > 1) { mainDivs[1].scrollTop = mainDivs[1].scrollHeight; }
                """)
        
        time.sleep(3.5) 

        # --- ÉP BUNG FULL TEXT VÀ BẢN GỐC ---
        driver.execute_script("""
            let btns = document.querySelectorAll('button');
            for(let btn of btns){
                let txt = (btn.innerText || "").toLowerCase();
                if(txt.includes('xem thêm') || txt.includes('more') || 
                   txt.includes('bản gốc') || txt.includes('original') || 
                   btn.classList.contains('w8Bnuf')) {
                    try { btn.click(); } catch(e) {}
                }
            }
        """)
        
        # BẮT BUỘC CHỜ 2 GIÂY để DOM load đủ chữ và tiếng Anh gốc từ server
        time.sleep(5) 

        items = driver.find_elements(By.CSS_SELECTOR, 'div[data-review-id]')
        
        # Bắt đầu đọc dữ liệu
        for item in items:
            if len(collected_data) >= 1000:
                break
            try:
                detail_full = item.find_element(By.CLASS_NAME, "wiI7pd").text
                
                # Bỏ qua nếu text rỗng hoặc VẪN CÒN chứa "..."
                if not detail_full or detail_full.endswith("..."):
                    continue
                    
                if detail_full in seen_texts:
                    continue
                    
                rating_raw = item.find_element(By.XPATH, './/span[contains(@aria-label, "sao") or contains(@aria-label, "star")]').get_attribute("aria-label")
                rating = int(rating_raw.split()[0])
                user = item.find_element(By.CLASS_NAME, "d4r55").text
                
                seen_texts.add(detail_full)
                collected_data.append({
                    "Username": user,
                    "Address": address_name,
                    "Rating": rating,
                    "Detail": detail_full
                })
            except:
                continue

        current_total = len(collected_data)
        print(f"  -> Đã gom được {current_total}/1000 đánh giá (Bản gốc, Full chữ)...")

        if current_total >= 1000:
            print(f"🎉 TUYỆT VỜI! Đã cào đủ 1000 đánh giá nguyên bản!")
            break

        if current_total == last_total:
            stuck_counter += 1
            if stuck_counter >= 10: 
                print("⚠️ Google Maps đã ngưng nhả dữ liệu. Sẽ chốt sổ với số lượng hiện tại.")
                break
        else:
            stuck_counter = 0

        last_total = current_total

    return collected_data

In [8]:
# ==========================================
# KHU VỰC CHẠY CHÍNH 
# ==========================================

# 🛑 ĐIỀN THÔNG TIN 1 ĐỊA ĐIỂM CỦA BẠN VÀO ĐÂY:
ten_dia_diem = "VinWonders Phu Quoc"
link_google_maps = "https://www.google.com/maps/place/VinWonders+Ph%C3%BA+Qu%E1%BB%91c/@10.3407345,103.8514725,17z/data=!4m8!3m7!1s0x31081fff0f370283:0xd36b94f9a85d30cd!8m2!3d10.3407345!4d103.8540474!9m1!1b1!16s%2Fg%2F11r8503s0c?entry=ttu&g_ep=EgoyMDI2MDMxMS4wIKXMDSoASAFQAw%3D%3D"
# THIẾT LẬP KỶ LUẬT THÉP CẤM DỊCH
options = Options()
options.add_argument("--disable-notifications")
options.add_argument("--disable-features=Translate") # Khóa mõm Google Translate hoàn toàn

prefs = {
  "translate":{"enabled":"False"}
}
options.add_experimental_option("prefs", prefs)

driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
driver.maximize_window() 

# Khởi chạy 1 lần duy nhất
all_raw_data = crawl_1000_reviews(driver, link_google_maps, ten_dia_diem)

driver.quit()

# ==========================================
# XUẤT FILE KHÔNG GIỚI HẠN
# ==========================================
print("\n📊 Đang tiến hành xóa trùng lặp và xuất file...")

if all_raw_data:
    df_all = pd.DataFrame(all_raw_data)
    # Giữ lại toàn bộ dữ liệu, chỉ xóa những dòng trùng nhau y hệt
    df_final = df_all.drop_duplicates(subset=['Address', 'Username', 'Detail'])

    # Đặt tên file xuất ra theo tên địa điểm cho dễ quản lý
    ten_file = f"Data_{ten_dia_diem.replace(' ', '_')}.csv"
    df_final.to_csv(ten_file, index=False, encoding='utf-8-sig')
    
    print(f"🎉 THÀNH CÔNG! Đã giữ lại trọn vẹn {len(df_final)} dòng ra file {ten_file}")
else:
    print("\n😢 Không lấy được dữ liệu nào.")


[VinWonders Phu Quoc] >>> Đang mở link...

🚨 ĐÃ MỞ XONG LINK: VinWonders Phu Quoc
👉 BƯỚC 1: Hãy qua cửa sổ Chrome, tự chọn bộ lọc (Mới nhất/Cao nhất/Thấp nhất).
👉 BƯỚC 2: Quay lại đây BẤM PHÍM ENTER để code quét.

🚀 Bắt đầu càn quét mục tiêu 1000 đánh giá (Chống lỗi '...', CẤM DỊCH)...
  -> Đã gom được 149/1000 đánh giá (Bản gốc, Full chữ)...
  -> Đã gom được 159/1000 đánh giá (Bản gốc, Full chữ)...
  -> Đã gom được 168/1000 đánh giá (Bản gốc, Full chữ)...
  -> Đã gom được 177/1000 đánh giá (Bản gốc, Full chữ)...
  -> Đã gom được 187/1000 đánh giá (Bản gốc, Full chữ)...
  -> Đã gom được 197/1000 đánh giá (Bản gốc, Full chữ)...
  -> Đã gom được 207/1000 đánh giá (Bản gốc, Full chữ)...
  -> Đã gom được 207/1000 đánh giá (Bản gốc, Full chữ)...
  -> Đã gom được 207/1000 đánh giá (Bản gốc, Full chữ)...
  -> Đã gom được 207/1000 đánh giá (Bản gốc, Full chữ)...
  -> Đã gom được 207/1000 đánh giá (Bản gốc, Full chữ)...
  -> Đã gom được 207/1000 đánh giá (Bản gốc, Full chữ)...
  -> Đã gom được